# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aniqaatiq842-commits/Flyrank-ML-INTERNSHIP/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)


### Lane

My lane is **content refresh prioritization**.

### ML task type

I would frame this as a **classification problem**.

The goal is to predict whether a content page is likely to be declining and therefore may need review or a refresh.

The model would take information about a page, such as its age, content type, and other available page-level signals, and predict whether the page belongs to the declining group.

The output would support a content team's refresh queue by helping them prioritize which pages to review first.


In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

### Target

My target would be **`is_declining_label`**.

I would define it as:

* `1` = the page is declining
* `0` = the page is not declining

The starter data already contains `trend_direction`, so the target can be derived from whether the trend direction is `"down"`.

I would not use `trend_direction` or `trend_pct` as model features because they are directly connected to the target and could cause target leakage.

This target is useful because the actual content decision is whether a page should be considered for a refresh or further review.


In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

My main success metric would be **Precision@50**.

This measures how many of the top 50 pages recommended by the model are actually declining.

For example, if the model recommends 50 pages and 40 of them are actually declining:

**Precision@50 = 40 / 50 = 0.80**

This metric makes sense for the content refresh problem because the content team has limited time and cannot review every page.

A useful model should therefore put relevant declining pages near the top of the refresh queue rather than simply predicting every page individually.


In [33]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

fatal: destination path 'flyrank-ml-internship-starter' already exists and is not an empty directory.


In [34]:
import os

print(os.listdir("/content"))

['.config', 'flyrank-ml-internship-starter', 'sample_data']


In [35]:
import os

for root, dirs, files in os.walk("/content/flyrank-ml-internship-starter"):
    for file in files:
        if file == "content_refresh_anonymized.csv":
            print(os.path.join(root, file))

/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv


In [36]:
import pandas as pd

df = pd.read_csv(
    "/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"
)

print(df.shape)
print(df.columns.tolist())

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [37]:
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [38]:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)
df[["trend_direction", "is_declining_label"]].head(10)

,trend_direction,is_declining_label
0,down,1
1,down,1
2,down,1
3,stable,0
4,down,1
5,down,1
6,down,1
7,stable,0
8,down,1
9,down,1


## 4. The unit of analysis, as a real dataframe

The unit of analysis for my problem is **one content page**.

Each row in the dataset represents one content page, identified by `content_id`. The columns describe different characteristics and performance signals for that page, such as content type, content age, recent impressions, clicks, search position, and trend.

I created a smaller dataframe to make this unit of analysis clear. Each row represents one page, while `is_declining_label` represents whether that page is currently classified as declining.

This page-level framing makes sense because the final output will be used to prioritize individual pages for content review or refresh.


In [39]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
unit_df = df[
    [
        "content_id",
        "client_id",
        "content_type",
        "content_age_days",
        "days_since_last_update",
        "impressions_last_30d",
        "clicks_last_30d",
        "avg_position",
        "trend_direction",
        "is_declining_label"
    ]
].head(10)

unit_df


,content_id,client_id,content_type,content_age_days,days_since_last_update,impressions_last_30d,clicks_last_30d,avg_position,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,187,20,578,2,10.6,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,25,2501,2,20.3,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,20,2382,1,36.5,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,22,3626,22,6.2,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,14,4211,10,44.0,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,20,617,0,8.5,down,1
6,content_9a34b442b552,client_8722616204,keyword article,90,20,1,0,7.0,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,22,636,1,21.2,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,20,5696,9,46.0,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,257,104,252,0,4.9,down,1


In [40]:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

## 5. Why ML beats a fixed rule here

A simple rule could be used to prioritize pages, for example, refreshing every page that is more than 300 days old. However, content age alone does not tell us whether a page actually needs a refresh.

A page can be old but still perform well, while a newer page can already be losing impressions or clicks. The dataset contains several signals that could be useful together, including recent impressions, clicks, sessions, content age, days since the last update, and average search position.

ML could learn patterns across these different signals instead of relying on one manually chosen threshold. This could help identify pages that are more likely to be declining and place them higher in the content refresh queue.

The goal is not to automatically decide that every predicted page must be refreshed. The model would provide a prioritized list that the content team can review and act on.


In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["is_declining_label"].value_counts()


,count
is_declining_label,
1,16262
0,13738


The target contains 16,262 declining pages and 13,738 non-declining pages out of 30,000 pages. This means the two classes are not extremely imbalanced, so classification is a reasonable framing for this problem.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

 #Self-check

| Question                             | My framing                                                                                  |
| ------------------------------------ | ------------------------------------------------------------------------------------------- |
| What type of ML problem is this?     | Classification                                                                              |
| What is the target?                  | `is_declining_label`                                                                        |
| What does one row represent?         | One content page                                                                            |
| What is the success metric?          | Precision@50                                                                                |
| What action does the output support? | Prioritizing pages for content review and possible refresh                                  |
| Why use ML?                          | Multiple page-level signals can be considered together instead of relying on one fixed rule |

### Main caution

I need to avoid target leakage when building the model. In particular, `trend_direction` and `trend_pct` should not be used as input features if they are used to create the declining target.
